<a href="https://colab.research.google.com/github/Maryam-Yaqoob/NLP-LAB/blob/main/FA23_BAI_025___Lab_5___Maryam_Yaqoob.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP Lab Task 5 — Solved

This notebook solves **Tasks 1–9** using **Python standard library only**.

Rules followed:
- lowercase all text
- add `<s>` and `</s>`
- vocabulary `V` contains all words plus `</s>`, but not `<s>`
- natural logarithm (`ln`) is used
- displayed decimal values are rounded to 4 places


In [26]:
train = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
    "the dog ate the bone"
]

test = [
    "the cat sat on the rug",
    "the fish sat on the mat"
]

print("Training sentences:", len(train))
print("Test sentences:", len(test))

Training sentences: 4
Test sentences: 2


## Task 1: Tokenization

In [27]:
def tokenize(sentence):
    return ["<s>"] + sentence.lower().split(" ") + ["</s>"]

train_tokens = [tokenize(s) for s in train]
test_tokens = [tokenize(s) for s in test]

print("Tokenized training sentences:")
for tokens in train_tokens:
    print(tokens)

print("\nTokenized test sentences:")
for tokens in test_tokens:
    print(tokens)

with_markers = sum(len(tokens) for tokens in train_tokens)
without_markers = sum(len(tokens) - 2 for tokens in train_tokens)

print("\nTotal training tokens with markers:", with_markers)
print("Total training tokens without markers:", without_markers)

Tokenized training sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']

Tokenized test sentences:
['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']

Total training tokens with markers: 30
Total training tokens without markers: 22


In [28]:
def build_vocab(sentences):
    vocab = set()
    for sentence in sentences:
        tokens = tokenize(sentence)
        for token in tokens:
            if token != "<s>":
                vocab.add(token)
    return vocab

vocab = build_vocab(train)
V = len(vocab)

print("V =", V)
print("Sorted vocabulary:")
print(sorted(vocab))

V = 11
Sorted vocabulary:
['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2: Generating N-grams

In [29]:
def get_ngrams(tokens, n):
    if n <= 0:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

first_tokens = train_tokens[0]

print("Unigrams:", get_ngrams(first_tokens, 1))
print("Bigrams:", get_ngrams(first_tokens, 2))
print("Trigrams:", get_ngrams(first_tokens, 3))

Unigrams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
Bigrams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
Trigrams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]


In [30]:
for label, corpus in [("TRAIN", train_tokens), ("TEST", test_tokens)]:
    print(f"\n{label}")
    for tokens in corpus:
        m = len(tokens)
        print("\nTokens:", tokens)
        for n in (1, 2, 3):
            ngrams = get_ngrams(tokens, n)
            expected = max(0, m - n + 1)
            assert len(ngrams) == expected
            print(f"n={n}: actual={len(ngrams)}, expected={expected}")

too_large_n = len(first_tokens) + 1
print("\nWhen n is larger than sentence length:")
print(get_ngrams(first_tokens, too_large_n))


TRAIN

Tokens: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
n=1: actual=8, expected=8
n=2: actual=7, expected=7
n=3: actual=6, expected=6

Tokens: ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
n=1: actual=7, expected=7
n=2: actual=6, expected=6
n=3: actual=5, expected=5

Tokens: ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
n=1: actual=8, expected=8
n=2: actual=7, expected=7
n=3: actual=6, expected=6

Tokens: ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
n=1: actual=7, expected=7
n=2: actual=6, expected=6
n=3: actual=5, expected=5

TEST

Tokens: ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
n=1: actual=8, expected=8
n=2: actual=7, expected=7
n=3: actual=6, expected=6

Tokens: ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']
n=1: actual=8, expected=8
n=2: actual=7, expected=7
n=3: actual=6, expected=6

When n is larger than sentence length:
[]


## Task 3: Counting N-grams

In [31]:
from collections import Counter

def corpus_ngram_counts(tokenized_sentences, n):
    counter = Counter()
    for tokens in tokenized_sentences:
        counter.update(get_ngrams(tokens, n))
    return counter

unigram_counts = corpus_ngram_counts(train_tokens, 1)
bigram_counts = corpus_ngram_counts(train_tokens, 2)
trigram_counts = corpus_ngram_counts(train_tokens, 3)

def print_sorted_counts(counter, title):
    print(f"\n{title}")
    for ngram, count in sorted(counter.items(), key=lambda x: (-x[1], x[0])):
        print(f"{ngram}: {count}")

print_sorted_counts(unigram_counts, "Unigram counts")
print_sorted_counts(bigram_counts, "Bigram counts")
print_sorted_counts(trigram_counts, "Trigram counts")

print("\n5 most common bigrams:")
for bg, count in sorted(bigram_counts.items(), key=lambda x: (-x[1], x[0]))[:5]:
    print(bg, count)

print("\nDistinct unigrams:", len(unigram_counts))
print("Distinct bigrams:", len(bigram_counts))
print("Distinct trigrams:", len(trigram_counts))


Unigram counts
('the',): 8
('</s>',): 4
('<s>',): 4
('ate',): 2
('cat',): 2
('dog',): 2
('on',): 2
('sat',): 2
('bone',): 1
('fish',): 1
('mat',): 1
('rug',): 1

Bigram counts
('<s>', 'the'): 4
('ate', 'the'): 2
('on', 'the'): 2
('sat', 'on'): 2
('the', 'cat'): 2
('the', 'dog'): 2
('bone', '</s>'): 1
('cat', 'ate'): 1
('cat', 'sat'): 1
('dog', 'ate'): 1
('dog', 'sat'): 1
('fish', '</s>'): 1
('mat', '</s>'): 1
('rug', '</s>'): 1
('the', 'bone'): 1
('the', 'fish'): 1
('the', 'mat'): 1
('the', 'rug'): 1

Trigram counts
('<s>', 'the', 'cat'): 2
('<s>', 'the', 'dog'): 2
('sat', 'on', 'the'): 2
('ate', 'the', 'bone'): 1
('ate', 'the', 'fish'): 1
('cat', 'ate', 'the'): 1
('cat', 'sat', 'on'): 1
('dog', 'ate', 'the'): 1
('dog', 'sat', 'on'): 1
('on', 'the', 'mat'): 1
('on', 'the', 'rug'): 1
('the', 'bone', '</s>'): 1
('the', 'cat', 'ate'): 1
('the', 'cat', 'sat'): 1
('the', 'dog', 'ate'): 1
('the', 'dog', 'sat'): 1
('the', 'fish', '</s>'): 1
('the', 'mat', '</s>'): 1
('the', 'rug', '</s>'): 1

In [32]:
all_counts = {
    1: unigram_counts,
    2: bigram_counts,
    3: trigram_counts
}

def count_of(ngram):
    return all_counts[len(ngram)].get(tuple(ngram), 0)

print("Seen bigram ('the', 'cat'):", count_of(("the", "cat")))
print("Unseen bigram ('fish', 'sat'):", count_of(("fish", "sat")))

Seen bigram ('the', 'cat'): 2
Unseen bigram ('fish', 'sat'): 0


## Task 4: Unigram Model
![image.png](attachment:image.png)

In [33]:
# N = total unigram tokens excluding <s>, but including </s>
# ye code har unigram ki probability calculate karke ek valid unigram probability model bana raha hai.
N = sum(
    count for (word,), count in unigram_counts.items()
    if word != "<s>"
)

def unigram_prob(w):
    return unigram_counts.get((w,), 0) / N

print("Word\tCount\tProbability")
prob_sum = 0.0

for w in sorted(vocab):
    p = unigram_prob(w)
    prob_sum += p
    print(f"{w}\t{unigram_counts[(w,)]}\t{p:.4f}")

print("\nSum =", round(prob_sum, 4))
assert abs(prob_sum - 1.0) < 1e-12

Word	Count	Probability
</s>	4	0.1538
ate	2	0.0769
bone	1	0.0385
cat	2	0.0769
dog	2	0.0769
fish	1	0.0385
mat	1	0.0385
on	2	0.0769
rug	1	0.0385
sat	2	0.0769
the	8	0.3077

Sum = 1.0


In [34]:
#Yeh code poore sentence ki probability unigram model se calculate kar raha hai.
def unigram_sentence_prob(sentence):
    tokens = tokenize(sentence)[1:]   # exclude <s>, include </s>
    p = 1.0
    for w in tokens:
        p *= unigram_prob(w)
    return p

for sentence in test:
    print(sentence, "->", f"{unigram_sentence_prob(sentence):.10f}")

the cat sat on the rug -> 0.0000002550
the fish sat on the mat -> 0.0000001275


In [35]:
#Yahan code yeh demonstrate kar raha hai ke unigram model word order ko ignore karta hai.
from itertools import permutations

words = test[0].lower().split()
orders = list(permutations(words))[:4]

print("Different word orders with unigram probabilities:")
for order in orders:
    sentence = " ".join(order)
    print(sentence, "->", f"{unigram_sentence_prob(sentence):.10f}")

Different word orders with unigram probabilities:
the cat sat on the rug -> 0.0000002550
the cat sat on rug the -> 0.0000002550
the cat sat the on rug -> 0.0000002550
the cat sat the rug on -> 0.0000002550


## Task 5: Bigram Model (MLE)

In [36]:
# Number of times a token occurs as a bigram context
context_counts = Counter()
for (prev, w), count in bigram_counts.items():
    context_counts[prev] += count

def bigram_prob(prev, w):
    denominator = context_counts.get(prev, 0)
    if denominator == 0:
        return 0.0
    return bigram_counts.get((prev, w), 0) / denominator

def next_word_distribution(prev):
    dist = {}
    denominator = context_counts.get(prev, 0)
    if denominator == 0:
        return dist

    for (p, w), count in bigram_counts.items():
        if p == prev:
            dist[w] = count / denominator
    return dict(sorted(dist.items()))

dist_the = next_word_distribution("the")

print("P(next | 'the'):")
for word, p in dist_the.items():
    print(f"{word}: {p:.4f}")

print("Sum =", round(sum(dist_the.values()), 4))
assert abs(sum(dist_the.values()) - 1.0) < 1e-12

P(next | 'the'):
bone: 0.1250
cat: 0.2500
dog: 0.2500
fish: 0.1250
mat: 0.1250
rug: 0.1250
Sum = 1.0


In [37]:
def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    p = 1.0
    for prev, w in get_ngrams(tokens, 2):
        p *= bigram_prob(prev, w)
    return p

for sentence in test:
    print(sentence, "->", f"{bigram_sentence_prob(sentence):.10f}")

the cat sat on the rug -> 0.0156250000
the fish sat on the mat -> 0.0000000000


In [38]:
def predict_next(prev):
    dist = next_word_distribution(prev)
    if not dist:
        return None

    best_prob = max(dist.values())
    tied = [word for word, p in dist.items() if p == best_prob]
    return sorted(tied)[0]

for prev in ["<s>", "the", "cat", "dog", "sat"]:
    print(f"{prev!r} -> {predict_next(prev)!r}")

'<s>' -> 'the'
'the' -> 'cat'
'cat' -> 'ate'
'dog' -> 'ate'
'sat' -> 'on'


## Task 6: Trigram Model

In [39]:
bigram_prefix_counts = Counter()
for (w1, w2, w3), count in trigram_counts.items():
    bigram_prefix_counts[(w1, w2)] += count

def trigram_prob(w1, w2, w3):
    denominator = bigram_prefix_counts.get((w1, w2), 0)
    if denominator == 0:
        return 0.0
    return trigram_counts.get((w1, w2, w3), 0) / denominator

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)

    # First real word uses bigram model: P(w1 | <s>)
    p = bigram_prob(tokens[0], tokens[1])

    # Remaining real words + </s> use trigram model
    for i in range(2, len(tokens)):
        p *= trigram_prob(tokens[i-2], tokens[i-1], tokens[i])

    return p

for sentence in test:
    print(sentence, "->", f"{trigram_sentence_prob(sentence):.10f}")

the cat sat on the rug -> 0.1250000000
the fish sat on the mat -> 0.0000000000


In [40]:
print("Model\tDistinct N-grams")
print(f"Unigram\t{len(unigram_counts)}")
print(f"Bigram\t{len(bigram_counts)}")
print(f"Trigram\t{len(trigram_counts)}")

Model	Distinct N-grams
Unigram	12
Bigram	18
Trigram	19


In [41]:
def unseen_ngrams(sentence, n):
    tokens = tokenize(sentence)
    counter = all_counts[n]
    return [
        ng for ng in get_ngrams(tokens, n)
        if counter.get(ng, 0) == 0
    ]

for sentence in test:
    ub = unseen_ngrams(sentence, 2)
    ut = unseen_ngrams(sentence, 3)

    print("\nSentence:", sentence)
    print("Unseen bigrams:", ub)
    print("Unseen trigrams:", ut)

    if len(ub) > len(ut):
        print("More unseen N-grams: Bigram model")
    elif len(ut) > len(ub):
        print("More unseen N-grams: Trigram model")
    else:
        print("Both have the same number of unseen N-grams")


Sentence: the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
Both have the same number of unseen N-grams

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
More unseen N-grams: Trigram model


## Task 7: Zero-Probability Problem

In [42]:
def unseen_bigrams(sentence):
    return unseen_ngrams(sentence, 2)

for sentence in test:
    print("\nSentence:", sentence)
    print("Unseen bigrams:", unseen_bigrams(sentence))
    print("Bigram MLE sentence probability:", f"{bigram_sentence_prob(sentence):.10f}")


Sentence: the cat sat on the rug
Unseen bigrams: []
Bigram MLE sentence probability: 0.0156250000

Sentence: the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Bigram MLE sentence probability: 0.0000000000


In [43]:
words_only = sorted(vocab - {"</s>"})

total_possible_pairs = len(words_only) ** 2
seen_word_pairs = sum(
    1 for a in words_only for b in words_only
    if bigram_counts.get((a, b), 0) > 0
)

unseen_pairs = total_possible_pairs - seen_word_pairs
unseen_percentage = (unseen_pairs / total_possible_pairs) * 100

print("Words only:", len(words_only))
print("Possible word pairs:", total_possible_pairs)
print("Seen word pairs:", seen_word_pairs)
print("Unseen word pairs:", unseen_pairs)
print("Unseen percentage:", f"{unseen_percentage:.4f}%")

Words only: 10
Possible word pairs: 100
Seen word pairs: 13
Unseen word pairs: 87
Unseen percentage: 87.0000%


In [44]:
import math

def safe_log(p):
    return -math.inf if p == 0 else math.log(p)

second_prob = bigram_sentence_prob(test[1])
second_log_prob = safe_log(second_prob)

print("Second test sentence probability:", second_prob)
print("Log probability:", second_log_prob)

Second test sentence probability: 0.0
Log probability: -inf


## Task 8: Smoothing

In [45]:
def bigram_prob_addone(prev, w):
    return (bigram_counts.get((prev, w), 0) + 1) / (context_counts.get(prev, 0) + V)

pairs = [("fish", "sat"), ("the", "cat")]

print("Probability\tMLE\tAdd-one")
for prev, w in pairs:
    print(
        f"P({w}|{prev})\t"
        f"{bigram_prob(prev, w):.4f}\t"
        f"{bigram_prob_addone(prev, w):.4f}"
    )

Probability	MLE	Add-one
P(sat|fish)	0.0000	0.0833
P(cat|the)	0.2500	0.1579


In [46]:
# Valid previous-word contexts: <s> plus ordinary words.
# </s> is not used as a previous word because it ends a sentence.
possible_prev_words = ["<s>"] + sorted(vocab - {"</s>"})

for prev in possible_prev_words:
    total = sum(bigram_prob_addone(prev, w) for w in vocab)
    assert abs(total - 1.0) < 1e-12

print("All add-one next-word distributions sum to 1.")

All add-one next-word distributions sum to 1.


In [47]:
def bigram_prob_addk(prev, w, k):
    return (bigram_counts.get((prev, w), 0) + k) / (context_counts.get(prev, 0) + k * V)

print("k\tP(sat|fish)")
for k in [1, 0.5, 0.1, 0.01]:
    print(f"{k}\t{bigram_prob_addk('fish', 'sat', k):.4f}")

k	P(sat|fish)
1	0.0833
0.5	0.0769
0.1	0.0476
0.01	0.0090


## Task 9: Perplexity and Model Comparison

In [48]:
def perplexity(sentences, prob_fn):
    log_sum = 0.0
    M = 0

    for sentence in sentences:
        probs = prob_fn(sentence)

        for p in probs:
            if p == 0:
                return math.inf
            log_sum += math.log(p)
            M += 1

    return math.exp(-log_sum / M)

def bigram_mle_token_probs(sentence):
    tokens = tokenize(sentence)
    return [
        bigram_prob(prev, w)
        for prev, w in get_ngrams(tokens, 2)
    ]

def bigram_addone_token_probs(sentence):
    tokens = tokenize(sentence)
    return [
        bigram_prob_addone(prev, w)
        for prev, w in get_ngrams(tokens, 2)
    ]

def unigram_prob_addone(w):
    return (unigram_counts.get((w,), 0) + 1) / (N + V)

def unigram_addone_token_probs(sentence):
    tokens = tokenize(sentence)[1:]  # exclude <s>, include </s>
    return [unigram_prob_addone(w) for w in tokens]

results = [
    ("Bigram MLE", perplexity(test, bigram_mle_token_probs)),
    ("Bigram Add-one", perplexity(test, bigram_addone_token_probs)),
    ("Unigram Add-one", perplexity(test, unigram_addone_token_probs))
]

print("Model\tPerplexity")
for name, pp in results:
    value = "inf" if math.isinf(pp) else f"{pp:.4f}"
    print(f"{name}\t{value}")

Model	Perplexity
Bigram MLE	inf
Bigram Add-one	5.7969
Unigram Add-one	9.1370


In [49]:
def make_bigram_addk_prob_fn(k):
    def token_probs(sentence):
        tokens = tokenize(sentence)
        return [
            bigram_prob_addk(prev, w, k)
            for prev, w in get_ngrams(tokens, 2)
        ]
    return token_probs

k_values = [1, 0.5, 0.1, 0.01]
test_k_results = []

print("k\tTest Perplexity")
for k in k_values:
    pp = perplexity(test, make_bigram_addk_prob_fn(k))
    test_k_results.append((k, pp))
    print(f"{k}\t{pp:.4f}")

best_test_k, best_test_pp = min(test_k_results, key=lambda x: x[1])

print("\nBest k on test set:", best_test_k)
print("Best test perplexity:", f"{best_test_pp:.4f}")

k	Test Perplexity
1	5.7969
0.5	4.5703
0.1	2.9559
0.01	2.6272

Best k on test set: 0.01
Best test perplexity: 2.6272


In [50]:
# Train only on first 3 training sentences, but keep V from all 4 sentences.
train3_tokens = [tokenize(s) for s in train[:3]]

bigram_counts_3 = corpus_ngram_counts(train3_tokens, 2)

context_counts_3 = Counter()
for (prev, w), count in bigram_counts_3.items():
    context_counts_3[prev] += count

def bigram_prob_addk_3(prev, w, k):
    return (bigram_counts_3.get((prev, w), 0) + k) / (context_counts_3.get(prev, 0) + k * V)

def make_bigram_addk_prob_fn_3(k):
    def token_probs(sentence):
        tokens = tokenize(sentence)
        return [
            bigram_prob_addk_3(prev, w, k)
            for prev, w in get_ngrams(tokens, 2)
        ]
    return token_probs

validation_sentence = [train[3]]

validation_results = []
print("k\tValidation Perplexity")

for k in k_values:
    pp = perplexity(validation_sentence, make_bigram_addk_prob_fn_3(k))
    validation_results.append((k, pp))
    print(f"{k}\t{pp:.4f}")

best_validation_k, best_validation_pp = min(validation_results, key=lambda x: x[1])

test_pp_with_validation_k = perplexity(
    test,
    make_bigram_addk_prob_fn_3(best_validation_k)
)

print("\nBest k selected on validation sentence:", best_validation_k)
print("Validation perplexity:", f"{best_validation_pp:.4f}")
print("Test perplexity using validation-selected k:", f"{test_pp_with_validation_k:.4f}")

print("\nComparison")
print("Best k chosen directly on test set:", best_test_k)
print("Best k chosen on validation set:", best_validation_k)

if best_test_k == best_validation_k:
    print("The two procedures selected the same k.")
else:
    print("The two procedures selected different k values.")

k	Validation Perplexity
1	8.5857
0.5	8.0163
0.1	8.0260
0.01	13.1304

Best k selected on validation sentence: 0.5
Validation perplexity: 8.0163
Test perplexity using validation-selected k: 4.4539

Comparison
Best k chosen directly on test set: 0.01
Best k chosen on validation set: 0.5
The two procedures selected different k values.


## Final Observation

- The **bigram MLE model** can produce zero probability when a test bigram was never seen during training.
- **Add-k smoothing** removes this zero-probability problem.
- **Perplexity** is lower when the model assigns higher probability to the observed test sequence.
- Choosing `k` on a separate validation sentence is methodologically better than choosing it directly on the test set.
